# Week 2 · Lab 01
## Local API (`requests`) + SQL Extraction to pandas

> **AI Engineering Academy** · Gamut Technology Services

Data engineering for LLM pipelines starts with **getting data out** — from REST
APIs and from SQL databases — reliably and reproducibly. In this lab you'll extract
the *same* synthetic dataset two ways, then transform, join, and validate it into a
clean artifact an LLM pipeline could consume.

### ⚙️ No external internet required
The setup cell builds a synthetic **Cordwell Home & Hardware** SQLite database and
starts a small **FastAPI** server (`lab_api.py`) on `http://127.0.0.1:8000` that
serves it. The API and the SQL half read the **same** `cordwell.db`. See
`API_REFERENCE.md` and `cordwell_data_dictionary.md` for the endpoints and schema.

### Learning objectives
1. Call a local REST API with **query parameters**, parse JSON, and handle status codes robustly.
2. Implement **cursor pagination** and an **exponential-backoff** retry policy that respects `429` (via `Retry-After`) and `5xx`.
3. Extract from **SQLite** into pandas with **parameterized queries** and `pd.read_sql_query`, including **chunked reads**.
4. **Join, validate, and profile** the data, then persist it to **Parquet** and **JSONL** for downstream LLM use.

### Time budget — ~110 min
| Segment | Time |
|---|---|
| Setup (build DB + start API) | 8 min |
| **A.** HTTP API extraction with `requests` | 40 min |
| **B.** SQL → pandas (params, chunking, Parquet) | 35 min |
| **C.** Transform · join · validate → LLM-ready | 22 min |
| Wrap-up | 5 min |

### Files beside this notebook
- `build_cordwell_db.py` — the database generator (run for you in setup).
- `lab_api.py` — the local API (started for you in setup).
- `API_REFERENCE.md`, `cordwell_data_dictionary.md` — reference docs.


In [ ]:
%pip install -r requirements.txt

In [ ]:
# --- Setup: build the local database, start the local API ------------------
# You make NO external network calls in this lab. A generator script builds a
# synthetic SQLite database (Cordwell Home & Hardware), and a small FastAPI app
# serves that SAME database on localhost. One dataset, two access paths:
#   * Part A talks to the local REST API with `requests`
#   * Part B/C read the SQLite file directly with pandas
import os, time, json, sqlite3
from pathlib import Path
import requests
import pandas as pd

import build_cordwell_db          # the generator (shipped beside this notebook)
from lab_api import start_server   # the local API (shipped beside this notebook)

# 1) Build the database if it is not already present (seeded / reproducible).
DB_PATH = "cordwell.db"
if not Path(DB_PATH).exists():
    stats = build_cordwell_db.build(DB_PATH, n_orders=10_000, seed=2025)
    print("Built database:", stats)
else:
    print("Database already present:", DB_PATH)

# 2) Point the API at the database and start it on a background thread.
os.environ["CORDWELL_DB"] = DB_PATH
BASE_URL = "http://127.0.0.1:8000"
server, _thread = start_server(port=8000)
for _ in range(50):
    try:
        if requests.get(f"{BASE_URL}/health", timeout=1).status_code == 200:
            break
    except requests.exceptions.RequestException:
        time.sleep(0.1)
print("Local API ready:", requests.get(f"{BASE_URL}/health", timeout=2).json())

def check(label, predicate):
    """Soft self-check: prints PASS/FAIL, never raises."""
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("ready.")

## Part A — HTTP API extraction with `requests`

The API exposes the orders table with **keyset (cursor) pagination** and optional
filters. Endpoint:

```
GET /v1/orders?limit=<int>&cursor=<int>&region=<str>&channel=<str>
    -> {"data": [order, ...], "next_cursor": <int|null>, "count": <int>}
```

`next_cursor` is the last `order_id` of the page (pass it back as `cursor`), and it
is `null` on the final page.


### A1 — Warm-up: GET with params & `.json()`  *(guided)*

Always pass a **`timeout`**, and read the response envelope before assuming its shape.


In [ ]:
resp = requests.get(f"{BASE_URL}/v1/orders", params={"limit": 3}, timeout=10)
print("status:", resp.status_code)
payload = resp.json()
print("envelope keys:", list(payload.keys()))
print("count:", payload["count"], "| next_cursor:", payload["next_cursor"])
payload["data"][0]

### A2 — A robust request helper with retry + backoff

Real extraction code must survive transient failures. Implement
`get_json(url, params=None, *, max_retries=5)` that:

- issues a `GET` with a **timeout**;
- on **200**, returns the parsed JSON (`resp.json()`);
- on **429 / 500 / 502 / 503 / 504**, waits and retries with **exponential backoff**,
  and if the response carries a **`Retry-After`** header, waits *that many seconds*
  instead;
- on any **other 4xx**, raises `APIError` immediately (not retryable);
- on a `Timeout`/`ConnectionError`, waits and retries;
- after `max_retries`, raises `APIError`.

The API gives you two endpoints to prove it works: `/v1/unreliable` (503s a set number
of times, then 200) and `/v1/rate-limited` (429 + `Retry-After: 2`, then 200).


💡 **Hint.** Start `backoff = 0.3`. Loop `for attempt in range(1, max_retries+1)`. Wrap
the `requests.get` in `try/except (requests.Timeout, requests.ConnectionError)`. For a
retryable status, compute the wait as `float(resp.headers["Retry-After"])` when the
header is present, else `backoff`, then `time.sleep(wait)` and `backoff *= 2`. Reset the
server counters with `requests.post(f"{BASE_URL}/admin/reset")` before testing.


In [ ]:
class APIError(Exception):
    pass

def get_json(url, params=None, *, max_retries=5, timeout=15):
    backoff = 0.3
    # TODO: loop up to max_retries:
    #   - GET (handle Timeout/ConnectionError by sleeping + backing off)
    #   - 200 -> return resp.json()
    #   - 429/5xx -> sleep (Retry-After if present, else backoff), backoff *= 2, continue
    #   - other 4xx -> raise APIError immediately
    # after the loop: raise APIError(...)
    raise NotImplementedError

# quick manual checks (uncomment as you go):
# requests.post(f"{BASE_URL}/admin/reset", timeout=5)
# print(get_json(f"{BASE_URL}/v1/unreliable", {"key": "a2", "fail_times": 2}))

In [ ]:
# Recovers from transient 503s (2 failures -> success on attempt 3)
requests.post(f"{BASE_URL}/admin/reset", timeout=5)
_u = get_json(f"{BASE_URL}/v1/unreliable", {"key": "chkA2", "fail_times": 2})
check("A2: recovered from 503s via backoff (attempts == 3)", lambda: _u["attempts"] == 3)

# Honors Retry-After on a 429
requests.post(f"{BASE_URL}/admin/reset", timeout=5)
_r = get_json(f"{BASE_URL}/v1/rate-limited", {"key": "chkA2rl"})
check("A2: recovered from 429 (attempts == 2)", lambda: _r["attempts"] == 2)

# Non-retryable 4xx raises immediately
def _expect_apierror():
    try:
        get_json(f"{BASE_URL}/v1/does-not-exist")
        return False
    except APIError:
        return True
check("A2: non-retryable 4xx raises APIError", _expect_apierror)

### A3 — Pagination: assemble the full orders table
Write `fetch_all_orders(base_url, page_size=1000, **filters)` that walks the cursor
pagination using your `get_json` and returns a single `pd.DataFrame` of every order.
Start with `cursor=0`; after each page, read `next_cursor` and pass it back as `cursor`;
stop when `next_cursor` is `None`. Forward any `filters` (like `region=...`) as params.


💡 **Hint.** Keep a `params = {"limit": page_size, "cursor": 0, **filters}`. Each loop:
`payload = get_json(url, params)`, extend a list with `payload["data"]`, then
`if payload["next_cursor"] is None: break` else `params["cursor"] = payload["next_cursor"]`.
Build the DataFrame at the end with `pd.DataFrame(rows)`.


In [ ]:
def fetch_all_orders(base_url, page_size=1000, **filters):
    rows = []
    params = {"limit": page_size, "cursor": 0, **filters}
    # TODO: loop pages via cursor until next_cursor is None; extend rows
    return pd.DataFrame(rows)

orders_df = fetch_all_orders(BASE_URL)
print("fetched orders:", orders_df.shape)
orders_df.head()

In [ ]:
check("A3: fetched all 10,000 orders", lambda: len(orders_df) == 10_000)
check("A3: order_ids are unique", lambda: orders_df["order_id"].is_unique)
check("A3: has the expected columns",
      lambda: {"order_id", "store_region", "channel", "order_date"} <= set(orders_df.columns))

### A4 — Server-side filtering via query params
Fetching everything and filtering in pandas wastes bandwidth. The API filters
server-side: pass `region=` (and/or `channel=`). Fetch **only** the `Southeast` orders
into `southeast_df`, and cross-check the count against the database directly.


💡 **Hint.** Reuse `fetch_all_orders(BASE_URL, region="Southeast")`. For the cross-check,
`sqlite3.connect(DB_PATH)` and `SELECT COUNT(*) FROM orders WHERE store_region='Southeast'`.
The API count and the SQL count should match exactly.


In [ ]:
southeast_df = None   # TODO: fetch_all_orders(..., region="Southeast")

# cross-check against the database
_con = sqlite3.connect(DB_PATH)
sql_southeast = _con.execute(
    "SELECT COUNT(*) FROM orders WHERE store_region = ?", ("Southeast",)
).fetchone()[0]
_con.close()
print("API rows:", None if southeast_df is None else len(southeast_df), "| SQL count:", sql_southeast)

In [ ]:
check("A4: every returned order is in the Southeast region",
      lambda: (southeast_df["store_region"] == "Southeast").all())
check("A4: API filter count matches the database count",
      lambda: len(southeast_df) == sql_southeast)

## Part B — SQL → pandas with SQLite

Now read the database **directly**. `pd.read_sql_query` runs SQL and returns a
DataFrame; a raw `sqlite3` connection is all you need (no SQLAlchemy for SQLite).


### B1 — Parameterized queries (with a join)
Write SQL that joins `order_lines → orders → products` and returns line-level rows for
a given region **and** on/after a given date, then run it with **parameter binding**.
Fill `region` and `since` as Python values and pass them via `params=` — never with
f-strings or string concatenation.

Return columns: `order_id, order_date, store_region, channel, category, product_name,
quantity, unit_price, discount_pct`. Filter: `store_region = :region AND order_date >= :since`.


💡 **Hint.** Use `?` placeholders in the SQL and `params=(region, since)` in
`pd.read_sql_query(sql, conn, params=...)`. Join keys: `ol.order_id = o.order_id` and
`ol.product_id = p.product_id`. The table with the join is `order_lines ol`.


In [ ]:
conn = sqlite3.connect(DB_PATH)

region = "Southeast"
since = "2025-01-01"

sql = """
-- TODO: SELECT the requested columns
-- FROM order_lines ol
-- JOIN orders o   ON ...
-- JOIN products p ON ...
-- WHERE o.store_region = ? AND o.order_date >= ?
"""
lines_df = None   # TODO: pd.read_sql_query(sql, conn, params=(region, since))
print(None if lines_df is None else lines_df.shape)

In [ ]:
check("B1: returned some rows", lambda: len(lines_df) > 0)
check("B1: filter held (region)", lambda: (lines_df["store_region"] == "Southeast").all())
check("B1: filter held (date)", lambda: (lines_df["order_date"] >= "2025-01-01").all())
check("B1: join populated product columns",
      lambda: lines_df["category"].notna().all() and lines_df["product_name"].notna().all())

### B2 — Chunked reads → Parquet
The `order_lines` table is large (~45k rows here; imagine millions). Read it in
**chunks** so you never hold the whole table in memory, streaming each chunk to a single
Parquet file. Implement the streaming write, then confirm the row count round-trips.


💡 **Hint.** `pd.read_sql_query("SELECT * FROM order_lines", conn, chunksize=10_000)`
yields DataFrames. For each chunk, `pa.Table.from_pandas(chunk, preserve_index=False)`;
open a `pq.ParquetWriter(path, table.schema)` on the **first** chunk, then
`writer.write_table(table)` for every chunk; `writer.close()` at the end.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

OUT_PARQUET = "order_lines.parquet"
chunks = pd.read_sql_query("SELECT * FROM order_lines", conn, chunksize=10_000)

writer = None
rows_written = 0
# TODO: for each chunk -> Table.from_pandas -> open writer on first chunk -> write_table
# TODO: close the writer; track rows_written
print("rows written:", rows_written)

In [ ]:
_reloaded = pd.read_parquet(OUT_PARQUET)
_con = sqlite3.connect(DB_PATH)
_sql_count = _con.execute("SELECT COUNT(*) FROM order_lines").fetchone()[0]
_con.close()
check("B2: parquet file was written", lambda: Path(OUT_PARQUET).exists())
check("B2: rows_written matches the table row count", lambda: rows_written == _sql_count)
check("B2: parquet round-trips the same row count", lambda: len(_reloaded) == _sql_count)

### B3 — Validation snapshot & profiling
Before trusting extracted data, **profile** it and assert your expectations. Reload the
Parquet, build a small profile, and validate a few invariants.


💡 **Hint.** `df.describe()` for numeric profiling; `df.isnull().sum()` for nulls.
Invariants to assert: no nulls in `order_id`/`product_id`/`quantity`, `quantity >= 1`,
`unit_price > 0`, and `discount_pct` within `[0, 100]`.


In [ ]:
ol = pd.read_parquet(OUT_PARQUET)

profile = None       # TODO: ol.describe() (numeric summary)
null_counts = None   # TODO: ol.isnull().sum()

# invariants
quantity_ok = None   # TODO: (ol["quantity"] >= 1).all()
price_ok = None      # TODO: (ol["unit_price"] > 0).all()
discount_ok = None   # TODO: ol["discount_pct"].between(0, 100).all()
print(profile)

In [ ]:
check("B3: no nulls in key columns",
      lambda: null_counts[["line_id", "order_id", "product_id", "quantity"]].sum() == 0)
check("B3: quantity >= 1 everywhere", lambda: quantity_ok is True)
check("B3: unit_price > 0 everywhere", lambda: price_ok is True)
check("B3: discount_pct within [0, 100]", lambda: discount_ok is True)

## Part C — Transform · join · validate → LLM-ready

Extraction is only step one. To feed an LLM workflow you usually **join** the pieces
into a denormalized view, **validate** it, and **serialize** it to a format the pipeline
consumes (here, JSONL — one JSON object per line, the lingua franca of LLM data).

> **Responsible-AI note.** Data quality *is* a safety control: an LLM fine-tuned or
> grounded on malformed, inconsistent data produces worse, less-trustworthy output.
> The validation below is the quality gate that protects everything downstream.


### C1 — Build and validate the enriched line-level view
Join `order_lines → orders → products` for **all** data, add a computed
`line_total = quantity * unit_price * (1 - discount_pct/100)` (rounded to 2 dp), and
validate referential + range invariants.


💡 **Hint.** You can do the join in SQL (like B1 but no WHERE) or in pandas with
`merge`. Then `enriched["line_total"] = (enriched["quantity"] * enriched["unit_price"] *
(1 - enriched["discount_pct"]/100)).round(2)`. Referential invariant: no nulls in the
joined `category`/`product_name` (every FK resolved).


In [ ]:
sql_all = """
SELECT ol.line_id, o.order_id, o.store_region, o.channel, o.order_date,
       p.category, p.product_name,
       ol.quantity, ol.unit_price, ol.discount_pct
FROM order_lines ol
JOIN orders   o ON ol.order_id = o.order_id
JOIN products p ON ol.product_id = p.product_id
"""
enriched = pd.read_sql_query(sql_all, conn)

# TODO: add line_total (rounded to 2 dp)
# TODO: referential + range validation flags:
ref_ok = None        # no nulls in category / product_name (all FKs resolved)
total_ok = None      # all line_total > 0
print(None if enriched is None else enriched.shape)

In [ ]:
_con = sqlite3.connect(DB_PATH)
_line_count = _con.execute("SELECT COUNT(*) FROM order_lines").fetchone()[0]
_con.close()
check("C1: every order line survived the joins (referential integrity)",
      lambda: len(enriched) == _line_count)
check("C1: all foreign keys resolved (no null product fields)", lambda: ref_ok is True)
check("C1: line_total computed and positive", lambda: total_ok is True)
check("C1: line_total math is correct on row 0",
      lambda: abs(enriched.loc[0, "line_total"]
                  - round(enriched.loc[0, "quantity"] * enriched.loc[0, "unit_price"]
                          * (1 - enriched.loc[0, "discount_pct"]/100), 2)) < 0.01)

### C2 — Serialize an LLM-ready JSONL artifact
LLM ingestion/fine-tuning pipelines consume **JSONL** (one JSON object per line).
Aggregate `enriched` to **one record per order** and write JSONL where each record has
structured fields plus a natural-language `text` summary.

Each record: `{"order_id", "store_region", "channel", "order_date", "n_lines",
"order_total", "text"}` where `text` is a sentence like
*"Order 42 (Southeast, Online) on 2025-03-04: 3 line items totaling $87.45."*


💡 **Hint.** `grouped = enriched.groupby("order_id").agg(...)` to get per-order
`n_lines` (count), `order_total` (sum of line_total), and the first
region/channel/order_date. Reset the index, build the `text` with an f-string, then write
with `df.to_json(path, orient="records", lines=True)` — that's JSONL.


In [ ]:
grouped = (
    enriched.groupby("order_id")
    .agg(store_region=("store_region", "first"),
         channel=("channel", "first"),
         order_date=("order_date", "first"),
         n_lines=("line_id", "count"),
         order_total=("line_total", "sum"))
    .reset_index()
)
grouped["order_total"] = grouped["order_total"].round(2)

# TODO: build the natural-language `text` column (one sentence per order)
# TODO: write JSONL to "orders_llm.jsonl" with orient="records", lines=True
OUT_JSONL = "orders_llm.jsonl"
n_written = None
print(None if n_written is None else n_written)

In [ ]:
_records = [json.loads(line) for line in open(OUT_JSONL)]
check("C2: one JSONL record per order (10,000)", lambda: n_written == 10_000)
check("C2: JSONL is valid and parseable", lambda: len(_records) == 10_000)
check("C2: each record has the required fields",
      lambda: {"order_id", "store_region", "order_total", "text"} <= set(_records[0].keys()))
check("C2: the text summary is a non-empty string",
      lambda: isinstance(_records[0]["text"], str) and len(_records[0]["text"]) > 20)

## Wrap-up

You extracted the same dataset two ways (resilient REST calls and direct SQL),
transformed and validated it, and shipped Parquet + JSONL artifacts ready for an LLM
pipeline. Answer these in a markdown cell (they're the kind of thing a reviewer asks):

1. How does your retry/backoff behave for **429** vs **500** — what's the same, what's different?
2. Why are **parameterized** queries the default? Give a one-sentence example of an injection if they aren't used.
3. When would you choose **chunked** reads, and what trade-off do you accept?

**Stretch goals:** (a) add random **jitter** to the backoff sleep; (b) re-run B2 with
`dtype_backend="pyarrow"` and compare dtypes; (c) partition the JSONL by `store_region`.


In [1]:
# Clean shutdown of the local API.
conn.close()
server.should_exit = True
time.sleep(0.3)
print("Local API stopped. Artifacts:", [p for p in ["order_lines.parquet", "orders_llm.jsonl"] if Path(p).exists()])

NameError: name 'conn' is not defined